# Lesson 22 — AWS S3 Integration

## What you will learn
- Core **S3 operations**: upload, download, delete
- **Conversation snapshots**: save/load chat history to S3 per thread
- **Document storage**: upload PDFs/text for the RAG agent
- **Pre-signed URLs**: let frontend download directly (bypasses server)
- **GDPR erasure**: `erase_user_data()` deletes all S3 data for a user

## S3 Key structure
```
s3://your-bucket/
├── conversations/
│   └── {tenant_id}/
│       └── {thread_id}/
│           └── {timestamp}.json     ← conversation snapshot
└── documents/
    └── {tenant_id}/
        └── {filename}               ← RAG documents
```

In [ ]:
# !pip install boto3 langgraph langchain-ollama

## Step 1 — S3 Client with Simulation Fallback

In [ ]:
import os
import json
import time
import io
from datetime import datetime
from typing import Optional

S3_BUCKET          = os.getenv("S3_BUCKET_NAME", "my-langgraph-bucket")
CONV_PREFIX        = os.getenv("S3_CONVERSATION_PREFIX", "conversations")
DOCS_PREFIX        = os.getenv("S3_DOCUMENTS_PREFIX", "documents")
SIMULATION_MODE    = True  # Set to False with real AWS credentials

# In-memory simulation store
_sim_store: dict = {}

def create_s3_client():
    """Create S3 client — simulation if no AWS credentials."""
    if SIMULATION_MODE:
        return None
    try:
        import boto3
        session = boto3.Session(region_name=os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
        return session.client("s3")
    except Exception as e:
        print(f"[WARN] S3 unavailable: {e}")
        return None

s3 = create_s3_client()
print(f"S3 client: {'simulation mode' if SIMULATION_MODE else 'real AWS'}")

## Step 2 — Core S3 Operations

In [ ]:
def s3_upload_text(key: str, content: str) -> bool:
    if SIMULATION_MODE:
        _sim_store[key] = content
        print(f"  [S3-SIM] PUT s3://{S3_BUCKET}/{key} ({len(content)} chars)")
        return True
    try:
        s3.put_object(Bucket=S3_BUCKET, Key=key, Body=content.encode("utf-8"), ContentType="text/plain")
        return True
    except Exception as e:
        print(f"[ERROR] upload_text: {e}"); return False

def s3_download_text(key: str) -> Optional[str]:
    if SIMULATION_MODE:
        content = _sim_store.get(key)
        if content:
            print(f"  [S3-SIM] GET s3://{S3_BUCKET}/{key}")
        return content
    try:
        resp = s3.get_object(Bucket=S3_BUCKET, Key=key)
        return resp["Body"].read().decode("utf-8")
    except Exception as e:
        print(f"[ERROR] download_text: {e}"); return None

def s3_delete_object(key: str) -> bool:
    if SIMULATION_MODE:
        existed = key in _sim_store
        _sim_store.pop(key, None)
        print(f"  [S3-SIM] DELETE s3://{S3_BUCKET}/{key} (existed={existed})")
        return existed
    try:
        s3.delete_object(Bucket=S3_BUCKET, Key=key)
        return True
    except Exception as e:
        print(f"[ERROR] delete: {e}"); return False

def s3_list_keys(prefix: str) -> list:
    if SIMULATION_MODE:
        keys = [k for k in _sim_store if k.startswith(prefix)]
        return keys
    try:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix)
        return [obj["Key"] for obj in resp.get("Contents", [])]
    except Exception as e:
        print(f"[ERROR] list: {e}"); return []

# Quick test
s3_upload_text("test/hello.txt", "Hello, S3!")
print(s3_download_text("test/hello.txt"))
s3_delete_object("test/hello.txt")

## Step 3 — Conversation Snapshots

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

def save_conversation_to_s3(tenant_id: str, thread_id: str, messages: list) -> str:
    ts = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    key = f"{CONV_PREFIX}/{tenant_id}/{thread_id}/{ts}.json"
    data = [
        {"role": "human" if isinstance(m, HumanMessage) else "ai", "content": m.content}
        for m in messages
    ]
    s3_upload_text(key, json.dumps(data, indent=2))
    return key

def load_latest_conversation_from_s3(tenant_id: str, thread_id: str) -> list:
    prefix = f"{CONV_PREFIX}/{tenant_id}/{thread_id}/"
    keys = sorted(s3_list_keys(prefix))  # alphabetical = chronological
    if not keys:
        return []
    latest = s3_download_text(keys[-1])
    if not latest:
        return []
    data = json.loads(latest)
    return [
        HumanMessage(content=m["content"]) if m["role"] == "human" else AIMessage(content=m["content"])
        for m in data
    ]

# Test conversation snapshots
messages = [
    HumanMessage(content="What is LangGraph?"),
    AIMessage(content="LangGraph is a library for building stateful AI agent workflows."),
    HumanMessage(content="How does state work?"),
    AIMessage(content="State is a TypedDict shared between all nodes."),
]

key = save_conversation_to_s3("acme", "user-alice", messages)
print(f"Saved to: {key}")

loaded = load_latest_conversation_from_s3("acme", "user-alice")
print(f"Loaded {len(loaded)} messages:")
for m in loaded:
    role = "Human" if isinstance(m, HumanMessage) else "AI"
    print(f"  [{role}]: {m.content[:60]}")

## Step 4 — Document Storage for RAG

In [ ]:
def upload_document(tenant_id: str, filename: str, content: str) -> str:
    key = f"{DOCS_PREFIX}/{tenant_id}/{filename}"
    s3_upload_text(key, content)
    return key

def load_document_as_text(tenant_id: str, filename: str) -> Optional[str]:
    key = f"{DOCS_PREFIX}/{tenant_id}/{filename}"
    return s3_download_text(key)

def s3_generate_presigned_url(key: str, expires_in: int = 3600) -> str:
    if SIMULATION_MODE:
        return f"https://simulation.s3.amazonaws.com/{S3_BUCKET}/{key}?expires={expires_in}"
    try:
        return s3.generate_presigned_url(
            "get_object",
            Params={"Bucket": S3_BUCKET, "Key": key},
            ExpiresIn=expires_in,
        )
    except Exception as e:
        return f"ERROR: {e}"

# Test document operations
doc_key = upload_document("acme", "policy.txt", "All employees must use 2FA. Remote work allowed 3 days/week.")
print(f"Document uploaded: {doc_key}")
print(f"Content: {load_document_as_text('acme', 'policy.txt')}")
print(f"Pre-signed URL: {s3_generate_presigned_url(doc_key)[:80]}...")

## Step 5 — GDPR Erasure

In [ ]:
def erase_user_data(tenant_id: str, thread_id: str) -> dict:
    """Delete ALL S3 data for a user (GDPR right to be forgotten)."""
    deleted = []
    
    # Delete conversation history
    conv_prefix = f"{CONV_PREFIX}/{tenant_id}/{thread_id}/"
    for key in s3_list_keys(conv_prefix):
        if s3_delete_object(key):
            deleted.append(key)
    
    return {"deleted_keys": deleted, "count": len(deleted)}

# Test GDPR erasure
print("Keys before erasure:", s3_list_keys(f"{CONV_PREFIX}/acme/user-alice/"))
result = erase_user_data("acme", "user-alice")
print(f"Erased {result['count']} objects: {result['deleted_keys']}")
print("Keys after erasure:", s3_list_keys(f"{CONV_PREFIX}/acme/user-alice/"))

## Key Takeaways

| Operation | Key pattern |
|-----------|-------------|
| Upload text | `conversations/{tenant}/{thread}/{ts}.json` |
| Download latest | `list_keys(prefix)` → sort → download last |
| Document for RAG | `documents/{tenant}/{filename}` |
| Pre-signed URL | Temporary URL for frontend direct download |
| GDPR erasure | List all keys for user → delete each |
| Credentials | `boto3.Session()` picks best available automatically |

## 🏋️ Exercise
1. Add `list_user_threads(tenant_id)` that returns all unique thread IDs for a tenant from S3
2. Add conversation versioning: save every turn as a separate snapshot, then implement `get_turn(thread_id, turn_number)`
3. Set `SIMULATION_MODE = False`, add real AWS credentials to `.env`, and test against a real S3 bucket

In [ ]:
# Your exercise solution here
